# 09 · Reducción de características V5 · CV temporal
Objetivo: evaluar cuántas de las **192 características V5** necesitamos realmente.

Se prueban K = `40, 60, 80, 100, 120, 150, 192` para **Regresión Logística** y **SVM RBF**. La selección se hace dentro de cada fold usando solo su subentrenamiento, mediante información mutua. Los hiperparámetros permanecen congelados con los mejores valores de la optimización intensiva V5.

**No se usa 2018–2021 para escoger K. Tampoco se toca 2022–2025 ni holdout espacial.**


In [ ]:
%pip install -q scikit-learn joblib


In [ ]:
from pathlib import Path
import importlib, json, sys, warnings
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
warnings.filterwarnings('ignore')
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)


## 1. Localizar proyecto y cargar datos


In [ ]:
candidates=[Path.cwd(),Path.cwd()/'rain-threat-classifier',Path('/content/rain-threat-classifier'),Path('/content/drive/MyDrive/rain-threat-classifier')]
PROJECT_DIR=next((p for p in candidates if (p/'resultados_completo'/'dataset_modelo_mensual_v3.csv').exists() and (p/'04_climatologia_era5land.py').exists() and (p/'resultados_experimentos'/'v5_optimizacion_intensiva'/'mejores_parametros_finales.json').exists()),None)
if PROJECT_DIR is None: raise FileNotFoundError('No se encontró el proyecto o faltan archivos requeridos.')
if str(PROJECT_DIR) not in sys.path: sys.path.insert(0,str(PROJECT_DIR))
clim=importlib.reload(importlib.import_module('04_climatologia_era5land'))
DATA_DIR=PROJECT_DIR/'resultados_completo'
TUNING_DIR=PROJECT_DIR/'resultados_experimentos'/'v5_optimizacion_intensiva'
OUTPUT_DIR=PROJECT_DIR/'resultados_experimentos'/'v5_reduccion_caracteristicas'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(DATA_DIR/'dataset_modelo_mensual_v3.csv')
df['period_start']=pd.to_datetime(df['period_start'])
df['target_period_start']=pd.to_datetime(df['target_period_start'])
monthly=pd.read_csv(DATA_DIR/'indicadores_mensuales_todas_zonas.csv')
monthly['period_start']=pd.to_datetime(monthly['period_start'])
features_v3=[x.strip() for x in (DATA_DIR/'columnas_modelo_v3.txt').read_text(encoding='utf-8').splitlines() if x.strip()]
features_v5=features_v3+clim.CLIMATE_FEATURE_NAMES
best_params=json.loads((TUNING_DIR/'mejores_parametros_finales.json').read_text(encoding='utf-8'))
assert len(features_v5)==192
print('PROJECT_DIR:',PROJECT_DIR)
print('Features V5:',len(features_v5))


## 2. Construir folds temporales con climatología sin leakage


In [ ]:
train_df=df[df['split']=='entrenamiento'].copy()
le=LabelEncoder().fit(train_df['target_amenaza'])
CLASS_NAMES=list(le.classes_)
TEMPORAL_FOLDS=[('F1','2004-12-01','2005-01-01','2007-12-01'),('F2','2007-12-01','2008-01-01','2010-12-01'),('F3','2010-12-01','2011-01-01','2013-12-01'),('F4','2013-12-01','2014-01-01','2017-12-01')]
fold_data=[]
for name,train_end,val_start,val_end in TEMPORAL_FOLDS:
    tr=train_df[train_df['target_period_start']<=pd.Timestamp(train_end)].copy()
    va=train_df[train_df['target_period_start'].between(pd.Timestamp(val_start),pd.Timestamp(val_end))].copy()
    ref=clim.fit_climatology(monthly=monthly,cutoff=tr['period_start'].max(),zones=tr['zone_id'].unique())
    Xtr=clim.add_climate_features(tr,tr[features_v3].copy(),ref)
    Xva=clim.add_climate_features(va,va[features_v3].copy(),ref)
    fold_data.append({'name':name,'X_train':Xtr,'y_train':le.transform(tr['target_amenaza']),'X_val':Xva,'y_val':le.transform(va['target_amenaza'])})
    print(name,Xtr.shape,Xva.shape)


## 3. Modelos con hiperparámetros congelados


In [ ]:
models={
 'Regresion_Logistica':Pipeline([('scaler',StandardScaler()),('model',LogisticRegression(max_iter=5000,solver='lbfgs',random_state=RANDOM_STATE))]),
 'SVM_RBF':Pipeline([('scaler',StandardScaler()),('model',SVC(kernel='rbf',probability=False,cache_size=2048,random_state=RANDOM_STATE))])
}
for name in models:
    models[name].set_params(**best_params[name])
    print(name,best_params[name])


## 4. Ranking de información mutua dentro de cada fold


In [ ]:
fold_rankings={}
for fold in fold_data:
    mi=mutual_info_classif(fold['X_train'],fold['y_train'],random_state=RANDOM_STATE)
    ranking=pd.DataFrame({'feature':fold['X_train'].columns,'mi':mi}).sort_values(['mi','feature'],ascending=[False,True]).reset_index(drop=True)
    fold_rankings[fold['name']]=ranking
    ranking.to_csv(OUTPUT_DIR/f"ranking_mi_{fold['name']}.csv",index=False)
print('Rankings listos.')


## 5. Evaluar distintos tamaños K


In [ ]:
K_VALUES=[40,60,80,100,120,150,192]
rows=[]
for model_name,base_model in models.items():
    for k in K_VALUES:
        macro=[]; bal=[]; rec_alta=[]; f1_media=[]
        for fold in fold_data:
            selected=list(fold['X_train'].columns) if k==192 else fold_rankings[fold['name']].head(k)['feature'].tolist()
            model=clone(base_model)
            model.fit(fold['X_train'][selected],fold['y_train'])
            pred=model.predict(fold['X_val'][selected])
            p,r,f1,s=precision_recall_fscore_support(fold['y_val'],pred,labels=np.arange(len(CLASS_NAMES)),zero_division=0)
            macro.append(f1_score(fold['y_val'],pred,average='macro'))
            bal.append(balanced_accuracy_score(fold['y_val'],pred))
            rec_alta.append(r[CLASS_NAMES.index('Alta')])
            f1_media.append(f1[CLASS_NAMES.index('Media')])
        rows.append({'modelo':model_name,'k_features':k,'macro_f1_cv_mean':np.mean(macro),'macro_f1_cv_std':np.std(macro),'balanced_accuracy_cv_mean':np.mean(bal),'recall_alta_cv_mean':np.mean(rec_alta),'f1_media_cv_mean':np.mean(f1_media)})
        print(model_name,'K=',k,'Macro F1=',round(np.mean(macro),4),'Recall Alta=',round(np.mean(rec_alta),4))
results=pd.DataFrame(rows)
display(results.sort_values(['modelo','k_features']))
results.to_csv(OUTPUT_DIR/'comparacion_k_features_cv.csv',index=False)


## 6. Regla de 1 error estándar para recomendar un K compacto
Se elige el menor K cuyo Macro F1 quede dentro de un error estándar del mejor resultado medio del modelo.


In [ ]:
selection=[]
for model_name in results['modelo'].unique():
    sub=results[results['modelo']==model_name].copy()
    best=sub.loc[sub['macro_f1_cv_mean'].idxmax()]
    se=best['macro_f1_cv_std']/np.sqrt(len(TEMPORAL_FOLDS))
    threshold=best['macro_f1_cv_mean']-se
    chosen=sub[sub['macro_f1_cv_mean']>=threshold].sort_values('k_features').iloc[0]
    selection.append({'modelo':model_name,'k_mejor_media':int(best['k_features']),'mejor_macro_f1':best['macro_f1_cv_mean'],'error_estandar':se,'umbral_1se':threshold,'k_recomendado':int(chosen['k_features']),'macro_f1_k_recomendado':chosen['macro_f1_cv_mean'],'recall_alta_k_recomendado':chosen['recall_alta_cv_mean']})
recommended=pd.DataFrame(selection)
display(recommended)
recommended.to_csv(OUTPUT_DIR/'k_recomendado_regla_1se.csv',index=False)


## 7. Interpretación
Después de elegir K con esta CV interna, el siguiente paso será recalcular el ranking usando **todo 1991–2017**, seleccionar las K mejores características y entrenar la Regresión Logística completa para compararla en 2018–2021 contra el techo actual de **Macro F1 ≈ 0.4521**.

Todavía no usar 2022–2025 ni holdout espacial.
